# D6 — Diff D5 (CPRA-derived) vs HCD (mirror)

Compares the CPRA-derived APR (`output/D5/table_a2_CY{2018..2025}.csv`) against HCD's published APR mirror (`databases/hcd_apr_mirror.db.table_a2`), producing per-year diff CSVs and a headline summary.

## Outputs

- `output/D6/diff_CY{year}.csv` — one per year, sorted by category then street_address
- `output/D6/dedup_audit_CY2025.csv` — records of HCD CY 2025 dedup choices
- `output/D6/summary.csv` — headline counts per year
- `output/D6/methodology_notes.md` — auto-generated; documents dedup, join, and validation

## Caveats

1. **D5 is parcel-grouped (one row per APN); HCD is permit-row (one row per permit event).** The join therefore expands and contracts in interesting ways — a parcel with 3 permits in D5 will join to 3 HCD rows in different years.
2. **HCD has entitlement data (Table A); D5 has Work Type and structured Unit fields.** The diff is necessarily lossy in both directions for fields that exist in only one source.
3. **240 duplicate rows in HCD CY 2025 are dropped via exact-match criteria.** The audit CSV documents which. If Berkeley submits an updated CY 2025 to HCD, the duplicate pattern may resolve at the source; in that case rebuild the mirror via `scripts/build_hcd_mirror.py`.
4. **Income-tier unit columns in HCD are summed for total units comparison.** Berkeley's reporting to HCD includes affordability tier breakdowns CPRA doesn't capture.


## Cell 1 — Imports + paths

In [1]:
import pandas as pd
import sqlite3
import re
from pathlib import Path
from datetime import datetime

ROOT = Path("/Users/johngage/berkeley-data")
D5_DIR = ROOT / "output/D5"
HCD_DB = ROOT / "databases/hcd_apr_mirror.db"
OUT_DIR = ROOT / "output/D6"
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not HCD_DB.exists():
    raise FileNotFoundError(
        f"{HCD_DB} not found. Build it first: python scripts/build_hcd_mirror.py"
    )
for y in range(2018, 2026):
    p = D5_DIR / f"table_a2_CY{y}.csv"
    if not p.exists():
        raise FileNotFoundError(f"Missing D5 CSV: {p}. Run D5_apr_from_cpra.ipynb first.")
print(f"Paths OK. OUT_DIR: {OUT_DIR}")

Paths OK. OUT_DIR: /Users/johngage/berkeley-data/output/D6


## Cell 2 — Load HCD mirror (Berkeley only)

Reads `table_a2` filtered to `JURIS_NAME='BERKELEY'`. Casts date columns and the `YEAR` column.

In [2]:
con = sqlite3.connect(f"file:{HCD_DB}?mode=ro&immutable=1", uri=True)
hcd = pd.read_sql_query("SELECT * FROM table_a2 WHERE JURIS_NAME='BERKELEY'", con)
con.close()

hcd["YEAR"] = pd.to_numeric(hcd["YEAR"], errors="coerce").astype("Int64")
for c in ["ENT_APPROVE_DT1", "BP_ISSUE_DT1", "CO_ISSUE_DT1"]:
    if c in hcd.columns:
        hcd[c] = pd.to_datetime(hcd[c], errors="coerce")

print(f"Total Berkeley rows (pre-dedup): {len(hcd)}")
print(f"Years present: {sorted(hcd['YEAR'].dropna().unique().tolist())}")

Total Berkeley rows (pre-dedup): 2170
Years present: [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


## Cell 3 — Dedup HCD CY 2025

The CY 2025 doubling is documented in `scripts/build_hcd_mirror.py` (Berkeley submitted twice; HCD appended). Drop exact-match duplicates within (YEAR=2025, APN, STREET_ADDRESS, JURS_TRACKING_ID, BP_ISSUE_DT1, CO_ISSUE_DT1) groups. Keep the first row per cluster. Audit log preserves the choice.

In [3]:
hcd_2025 = hcd[hcd["YEAR"] == 2025].copy()
print(f"CY 2025 pre-dedup: {len(hcd_2025)} rows")

dup_groups = hcd_2025.groupby(
    ["APN", "STREET_ADDRESS", "JURS_TRACKING_ID", "BP_ISSUE_DT1", "CO_ISSUE_DT1"],
    dropna=False,
)
clusters_audit = []
keep_idx = set()
for key, grp in dup_groups:
    sorted_idx = sorted(grp.index)
    kept = sorted_idx[0]
    keep_idx.add(kept)
    if len(grp) > 1:
        for ix in sorted_idx:
            clusters_audit.append({
                "row_index": ix, "kept_or_dropped": "kept" if ix == kept else "dropped",
                "cluster_size": len(grp),
                "APN": key[0], "STREET_ADDRESS": key[1], "JURS_TRACKING_ID": key[2],
                "BP_ISSUE_DT1": key[3], "CO_ISSUE_DT1": key[4],
            })

audit_df = pd.DataFrame(clusters_audit)
audit_df.to_csv(OUT_DIR / "dedup_audit_CY2025.csv", index=False)

hcd = pd.concat([hcd[hcd["YEAR"] != 2025], hcd_2025.loc[sorted(keep_idx)]], ignore_index=True)
print(f"CY 2025 post-dedup: {(hcd['YEAR'] == 2025).sum()} rows")
print(f"HCD total post-dedup: {len(hcd)} rows")
print(f"Audit log: {len(audit_df)} rows ({len(clusters_audit)} dup-cluster members)")

CY 2025 pre-dedup: 474 rows
CY 2025 post-dedup: 234 rows
HCD total post-dedup: 1930 rows
Audit log: 474 rows (474 dup-cluster members)


## Cell 4 — Load D5 outputs

Read all 8 per-year CSVs, concatenate, tag each row with its source year.

In [4]:
d5_frames = []
for y in range(2018, 2026):
    df = pd.read_csv(D5_DIR / f"table_a2_CY{y}.csv")
    df["__source_year"] = y
    d5_frames.append(df)
d5 = pd.concat(d5_frames, ignore_index=True)
d5["BP_ISSUE_DT1"] = pd.to_datetime(d5["BP_ISSUE_DT1"], errors="coerce")
d5["CO_ISSUE_DT1"] = pd.to_datetime(d5["CO_ISSUE_DT1"], errors="coerce")

print(f"D5 rows (8 years concatenated): {len(d5)}")
print(f"Distinct D5 projects (by JURS_TRACKING_ID): {d5['JURS_TRACKING_ID'].nunique()}")

# Source-side per-year counts — used for summary (avoids the prior summary bug
# where post-merge counts undercounted because HCD rows often have null tracking)
d5_per_year = d5.groupby("__source_year").size().to_dict()
hcd_per_year = {int(k): v for k, v in hcd.groupby("YEAR").size().to_dict().items()}
print()
print("D5 per-year row counts:  ", d5_per_year)
print("HCD per-year row counts: ", hcd_per_year)

D5 rows (8 years concatenated): 5431
Distinct D5 projects (by JURS_TRACKING_ID): 4064

D5 per-year row counts:   {2018: 407, 2019: 387, 2020: 329, 2021: 412, 2022: 641, 2023: 928, 2024: 1156, 2025: 1171}
HCD per-year row counts:  {2018: 216, 2019: 256, 2020: 235, 2021: 260, 2022: 244, 2023: 257, 2024: 228, 2025: 234}


## Cell 5 — Normalize join keys

- `apn_norm`: lowercase, stripped
- `tracking_norm`: uppercase, stripped, `-REVxx`/`-DEFxx` suffix removed — matches D5's master selection
- `addr_norm`: lowercase, single-spaced, trailing punctuation stripped

In [5]:
REV = re.compile(r"-(?:REV|DEF)\d*$", re.IGNORECASE)
ADDR_TRAIL = re.compile(r"[\s,\.]+$")

def norm_tracking(s):
    if not isinstance(s, str): return None
    s = s.strip().upper()
    return REV.sub("", s) or None

def norm_apn(s):
    if not isinstance(s, str): return None
    return s.strip().lower() or None

def norm_addr(s):
    if not isinstance(s, str): return None
    s = re.sub(r"\s+", " ", s.strip().lower())
    return ADDR_TRAIL.sub("", s) or None

for df in (d5, hcd):
    df["apn_norm"] = df["APN"].apply(norm_apn)
    df["tracking_norm"] = df["JURS_TRACKING_ID"].apply(norm_tracking)
    df["addr_norm"] = df["STREET_ADDRESS"].apply(norm_addr)

print(f"D5  apn_norm populated: {d5['apn_norm'].notna().sum()}/{len(d5)}")
print(f"HCD apn_norm populated: {hcd['apn_norm'].notna().sum()}/{len(hcd)}")
print(f"D5  tracking_norm populated: {d5['tracking_norm'].notna().sum()}/{len(d5)}")
print(f"HCD tracking_norm populated: {hcd['tracking_norm'].notna().sum()}/{len(hcd)}")

D5  apn_norm populated: 5431/5431
HCD apn_norm populated: 1930/1930
D5  tracking_norm populated: 5431/5431
HCD tracking_norm populated: 1684/1930


## Cell 6 — Outer join + categorize

Outer-join on `(apn_norm, tracking_norm, year)`. Categories:

- `d5_only`: row in D5 with no matching HCD row
- `hcd_only`: row in HCD with no matching D5 row
- `in_both_clean`: matched on all three keys + BP/CO dates within 90 days + unit counts within 1
- `in_both_date_divergent`: matched but BP or CO dates differ by >90 days
- `in_both_unit_divergent`: matched but BP_units or CO_units differ by ≥1
- `in_both_tracking_mismatch`: same APN+year but D5 and HCD use different tracking IDs

Compute HCD's per-row total units as the sum across all 11 income-tier columns per stage (BP and CO).

In [6]:
def sum_unit_cols(row, prefix):
    cols = [c for c in row.index
            if c.startswith(prefix)
            and (c.endswith("_INCOME") or c.endswith("_INCOME_DR") or c.endswith("_INCOME_NDR") or c.endswith("INCOME_NDR"))]
    total = 0
    for c in cols:
        try:
            v = float(row[c])
            if not pd.isna(v): total += v
        except (ValueError, TypeError):
            pass
    return total

hcd["hcd_bp_units"] = hcd.apply(lambda r: sum_unit_cols(r, "BP_"), axis=1)
hcd["hcd_co_units"] = hcd.apply(lambda r: sum_unit_cols(r, "CO_"), axis=1)
hcd["year"] = hcd["YEAR"]
d5_keyed = d5.rename(columns={"__source_year": "year"})

left = d5_keyed[["apn_norm","tracking_norm","year","BP_ISSUE_DT1","CO_ISSUE_DT1",
                 "BP_ABOVE_MOD_INCOME","CO_ABOVE_MOD_INCOME",
                 "STREET_ADDRESS","APN","JURS_TRACKING_ID","UNIT_CAT","WORK_TYPE","ADU_FLAG","in_v2",
                 "bp_cycle","co_cycle","bp_in_projection_period","co_in_projection_period"]].rename(
    columns={"BP_ISSUE_DT1":"d5_bp_date","CO_ISSUE_DT1":"d5_co_date",
             "BP_ABOVE_MOD_INCOME":"d5_bp_units","CO_ABOVE_MOD_INCOME":"d5_co_units",
             "STREET_ADDRESS":"d5_street","APN":"d5_apn","JURS_TRACKING_ID":"d5_tracking",
             "UNIT_CAT":"d5_unit_cat","WORK_TYPE":"d5_work_type",
             "bp_cycle":"d5_bp_cycle","co_cycle":"d5_co_cycle",
             "bp_in_projection_period":"d5_bp_in_projection_period",
             "co_in_projection_period":"d5_co_in_projection_period"})
right = hcd[["apn_norm","tracking_norm","year","BP_ISSUE_DT1","CO_ISSUE_DT1",
             "hcd_bp_units","hcd_co_units","STREET_ADDRESS","APN","JURS_TRACKING_ID","UNIT_CAT"]].rename(
    columns={"BP_ISSUE_DT1":"hcd_bp_date","CO_ISSUE_DT1":"hcd_co_date",
             "STREET_ADDRESS":"hcd_street","APN":"hcd_apn","JURS_TRACKING_ID":"hcd_tracking",
             "UNIT_CAT":"hcd_unit_cat"})

merged = pd.merge(left, right, on=["apn_norm","tracking_norm","year"], how="outer", indicator=True)

def categorize(row):
    m = row["_merge"]
    if m == "left_only": return "d5_only"
    if m == "right_only": return "hcd_only"
    bp_d5, bp_hcd = row.get("d5_bp_date"), row.get("hcd_bp_date")
    co_d5, co_hcd = row.get("d5_co_date"), row.get("hcd_co_date")
    bp_delta = (bp_d5 - bp_hcd).days if (pd.notna(bp_d5) and pd.notna(bp_hcd)) else None
    co_delta = (co_d5 - co_hcd).days if (pd.notna(co_d5) and pd.notna(co_hcd)) else None
    if (bp_delta is not None and abs(bp_delta) > 90) or (co_delta is not None and abs(co_delta) > 90):
        return "in_both_date_divergent"
    bp_u_delta = (row.get("d5_bp_units") or 0) - (row.get("hcd_bp_units") or 0)
    co_u_delta = (row.get("d5_co_units") or 0) - (row.get("hcd_co_units") or 0)
    if abs(bp_u_delta) >= 1 or abs(co_u_delta) >= 1:
        return "in_both_unit_divergent"
    return "in_both_clean"

shared_apns_per_year = {}
for (apn, yr), grp in merged.groupby(["apn_norm", "year"]):
    if (grp["_merge"] == "left_only").any() and (grp["_merge"] == "right_only").any():
        shared_apns_per_year[(apn, yr)] = True

def categorize_with_tracking_mismatch(row):
    cat = categorize(row)
    if cat in ("d5_only","hcd_only") and shared_apns_per_year.get((row["apn_norm"], row["year"])):
        return "in_both_tracking_mismatch"
    return cat

merged["diff_category"] = merged.apply(categorize_with_tracking_mismatch, axis=1)
print("Diff category distribution:")
print(merged["diff_category"].value_counts().to_string())

Diff category distribution:
diff_category
d5_only                      3969
in_both_clean                 808
hcd_only                      455
in_both_unit_divergent        449
in_both_tracking_mismatch     423
in_both_date_divergent          1


## Cell 7 — Per-year diff CSVs

Compute delta columns and auto-notes; sort by category then street_address; write one CSV per year.

In [7]:
merged["bp_date_delta_days"] = merged.apply(
    lambda r: (r["d5_bp_date"] - r["hcd_bp_date"]).days
    if (pd.notna(r["d5_bp_date"]) and pd.notna(r["hcd_bp_date"])) else None, axis=1)
merged["co_date_delta_days"] = merged.apply(
    lambda r: (r["d5_co_date"] - r["hcd_co_date"]).days
    if (pd.notna(r["d5_co_date"]) and pd.notna(r["hcd_co_date"])) else None, axis=1)
merged["bp_units_delta"] = merged["d5_bp_units"].fillna(0) - merged["hcd_bp_units"].fillna(0)
merged["co_units_delta"] = merged["d5_co_units"].fillna(0) - merged["hcd_co_units"].fillna(0)

def auto_note(row):
    notes = []
    if row["diff_category"] == "d5_only" and row["d5_unit_cat"] == "ADU":
        notes.append("ADU permit only in D5")
    if row["diff_category"] == "hcd_only" and row.get("hcd_unit_cat") == "ADU":
        notes.append("ADU permit only in HCD")
    if row["diff_category"] == "in_both_tracking_mismatch":
        notes.append("Same APN+year, different tracking IDs")
    if row["diff_category"] == "in_both_unit_divergent":
        notes.append(f"Unit count differs (BP {row['bp_units_delta']:+.0f}, CO {row['co_units_delta']:+.0f})")
    if row["diff_category"] == "in_both_date_divergent":
        notes.append(f"Date differs >90d (BP {row['bp_date_delta_days']}, CO {row['co_date_delta_days']})")
    return "; ".join(notes)

merged["notes"] = merged.apply(auto_note, axis=1)
merged["street_address"] = merged["hcd_street"].fillna(merged["d5_street"])
merged["apn"] = merged["hcd_apn"].fillna(merged["d5_apn"])

DIFF_COLS = ["diff_category","year","street_address","apn",
             "d5_tracking","hcd_tracking",
             "d5_bp_date","hcd_bp_date","bp_date_delta_days",
             "d5_co_date","hcd_co_date","co_date_delta_days",
             "d5_bp_units","hcd_bp_units","bp_units_delta",
             "d5_co_units","hcd_co_units","co_units_delta",
             "d5_unit_cat","hcd_unit_cat","d5_work_type","in_v2",
             "d5_bp_cycle","d5_co_cycle","d5_bp_in_projection_period","d5_co_in_projection_period",
             "notes"]

for year in range(2018, 2026):
    sub = merged[merged["year"] == year][DIFF_COLS].sort_values(["diff_category","street_address"])
    out_path = OUT_DIR / f"diff_CY{year}.csv"
    sub.to_csv(out_path, index=False)
    print(f"CY{year}: {len(sub)} rows -> {out_path.name}")

CY2018: 482 rows -> diff_CY2018.csv
CY2019: 468 rows -> diff_CY2019.csv
CY2020: 386 rows -> diff_CY2020.csv
CY2021: 469 rows -> diff_CY2021.csv
CY2022: 885 rows -> diff_CY2022.csv
CY2023: 991 rows -> diff_CY2023.csv
CY2024: 1210 rows -> diff_CY2024.csv
CY2025: 1214 rows -> diff_CY2025.csv


## Cell 7b — Cycle-segmented summary (additive)

Aggregates the merged diff by `(year, unified_bp_cycle)`. The unified cycle uses the D5-side value when D5 has the row (left-only or in_both), and computes the cycle from HCD's `BP_ISSUE_DT1` for HCD-only rows. Output: `output/D6/cycle_segmented_summary.csv`.

In [8]:
# Cycle-segmented summary — additive aggregation across the merged diff.
# Uses scripts.housing_rules.classifiers.cycle_for_date to fill in HCD-only rows.

import sys
from pathlib import Path as _Path
_p = _Path.cwd()
while not (_p / "scripts" / "housing_rules").exists() and _p.parent != _p:
    _p = _p.parent
if not (_p / "scripts" / "housing_rules").exists():
    raise RuntimeError(f"Could not locate scripts/housing_rules/ from {_Path.cwd()}")
if str(_p) not in sys.path:
    sys.path.insert(0, str(_p))

from scripts.housing_rules.classifiers import cycle_for_date, is_projection_period

def _to_date(x):
    if x is None or pd.isna(x): return None
    if hasattr(x, "date"): return x.date()
    return x

# Compute HCD-side cycle (parallel to d5_bp_cycle from D5)
merged["hcd_bp_cycle"] = merged["hcd_bp_date"].apply(lambda x: cycle_for_date(_to_date(x)))
merged["hcd_bp_in_projection_period"] = merged["hcd_bp_date"].apply(lambda x: is_projection_period(_to_date(x)))

# Unified bp_cycle: prefer D5's value when present (in_both or d5_only rows),
# fall back to HCD's (hcd_only rows).
merged["unified_bp_cycle"] = merged["d5_bp_cycle"].fillna(merged["hcd_bp_cycle"])
merged["unified_bp_in_projection_period"] = merged["d5_bp_in_projection_period"].fillna(
    merged["hcd_bp_in_projection_period"]
).fillna(False).astype(bool)

# Aggregate by (year, unified_bp_cycle, diff_category)
seg = (merged
       .groupby(["year", "unified_bp_cycle", "diff_category"], dropna=False)
       .size()
       .unstack(fill_value=0)
       .reset_index())

# Ensure all diff_category columns exist even if absent in some year/cycle slice
for cat in ["d5_only", "hcd_only", "in_both_clean", "in_both_date_divergent",
            "in_both_unit_divergent", "in_both_tracking_mismatch"]:
    if cat not in seg.columns:
        seg[cat] = 0

seg["total"] = seg[["d5_only","hcd_only","in_both_clean",
                    "in_both_date_divergent","in_both_unit_divergent",
                    "in_both_tracking_mismatch"]].sum(axis=1)

# Reorder for readability
seg = seg[["year", "unified_bp_cycle",
           "d5_only", "hcd_only", "in_both_clean",
           "in_both_date_divergent", "in_both_unit_divergent",
           "in_both_tracking_mismatch", "total"]].rename(columns={"unified_bp_cycle":"bp_cycle"})

seg.to_csv(OUT_DIR / "cycle_segmented_summary.csv", index=False)
print(f"Wrote cycle_segmented_summary.csv: {len(seg)} rows")
print()
print(seg.to_string(index=False))


Wrote cycle_segmented_summary.csv: 17 rows

 year bp_cycle  d5_only  hcd_only  in_both_clean  in_both_date_divergent  in_both_unit_divergent  in_both_tracking_mismatch  total
 2018      5th      155        17             56                       0                      38                          4    270
 2018      NaN      108        54             33                       0                      14                          3    212
 2019      5th      100        17             67                       0                      38                          7    229
 2019      NaN      107        59             40                       0                      30                          3    239
 2020      5th       83        22             86                       0                      30                          2    223
 2020      NaN       66        33             37                       0                      25                          2    163
 2021      5th      132        16      

/var/folders/zr/1lcy71z97n33bq1zyg3vtbp80000gn/T/ipykernel_49620/1827547874.py:28: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged["unified_bp_in_projection_period"] = merged["d5_bp_in_projection_period"].fillna(


## Cell 8 — Headline summary

Per-year counts using **source-side row counts** (not post-merge counts) to avoid undercounting when HCD's JURS_TRACKING_ID is null.

In [9]:
summary_rows = []
for year in range(2018, 2026):
    sub = merged[merged["year"] == year]
    summary_rows.append({
        "year": year,
        "d5_total_rows": d5_per_year.get(year, 0),
        "hcd_total_rows": int(hcd_per_year.get(year, 0)),
        "d5_only_count": int((sub["diff_category"] == "d5_only").sum()),
        "hcd_only_count": int((sub["diff_category"] == "hcd_only").sum()),
        "in_both_clean_count": int((sub["diff_category"] == "in_both_clean").sum()),
        "in_both_date_divergent_count": int((sub["diff_category"] == "in_both_date_divergent").sum()),
        "in_both_unit_divergent_count": int((sub["diff_category"] == "in_both_unit_divergent").sum()),
        "in_both_tracking_mismatch_count": int((sub["diff_category"] == "in_both_tracking_mismatch").sum()),
        # Cycle counts (additive per Phase B)
        "bp_cycle_5th_count": int((sub["unified_bp_cycle"] == "5th").sum()),
        "bp_cycle_6th_count": int((sub["unified_bp_cycle"] == "6th").sum()),
        "projection_period_count": int(sub["unified_bp_in_projection_period"].fillna(False).sum()),
        "d5_total_units": float(sub["d5_bp_units"].fillna(0).sum() + sub["d5_co_units"].fillna(0).sum()),
        "hcd_total_units": float(sub["hcd_bp_units"].fillna(0).sum() + sub["hcd_co_units"].fillna(0).sum()),
    })
summary_df = pd.DataFrame(summary_rows)
summary_df["unit_total_delta"] = summary_df["d5_total_units"] - summary_df["hcd_total_units"]
summary_df.to_csv(OUT_DIR / "summary.csv", index=False)

print("\nHeadline summary:")
print(summary_df.to_string(index=False))


Headline summary:
 year  d5_total_rows  hcd_total_rows  d5_only_count  hcd_only_count  in_both_clean_count  in_both_date_divergent_count  in_both_unit_divergent_count  in_both_tracking_mismatch_count  bp_cycle_5th_count  bp_cycle_6th_count  projection_period_count  d5_total_units  hcd_total_units  unit_total_delta
 2018            407             216            263              71                   89                             0                            52                                7                 270                   0                        0           224.0            609.0            -385.0
 2019            387             256            207              76                  107                             0                            68                               10                 229                   0                        0           449.0            676.0            -227.0
 2020            329             235            149              55                  12

## Cell 9 — Spot check against 9 verified addresses

For each of the 9 addresses validated via Chrome verification, report which diff category it lands in and whether dates/tracking/units align.

In [10]:
import re as _re
_REV = _re.compile(r"-(?:REV|DEF)\d*$", _re.IGNORECASE)
def _base(s):
    if not isinstance(s, str): return None
    return _REV.sub("", s.strip().upper())

merged["d5_tracking_base"] = merged["d5_tracking"].apply(_base)
merged["hcd_tracking_base"] = merged["hcd_tracking"].apply(_base)

VERIFIED = {
    "B2021-02225": "2650 Telegraph",
    "B2022-05117": "2440 Shattuck",
    "B2018-05067": "2556 Telegraph",
    "B2014-05752": "1698 University",
    "B2024-01924": "1598 University",
    "B2023-02332": "2538 Durant",
    "B2017-02610": "2067 University",
    "B2025-02283": "0 Virginia",
}
for permit, addr in VERIFIED.items():
    sub = merged[(merged["d5_tracking_base"] == permit) | (merged["hcd_tracking_base"] == permit)]
    print(f"\n{permit} ({addr}): {len(sub)} diff-row(s)")
    for _, r in sub.iterrows():
        bp_d5 = "NaN" if pd.isna(r["d5_bp_units"]) else f"{r['d5_bp_units']:.0f}"
        bp_hcd = "NaN" if pd.isna(r["hcd_bp_units"]) else f"{r['hcd_bp_units']:.0f}"
        co_d5 = "NaN" if pd.isna(r["d5_co_units"]) else f"{r['d5_co_units']:.0f}"
        co_hcd = "NaN" if pd.isna(r["hcd_co_units"]) else f"{r['hcd_co_units']:.0f}"
        print(f"  YEAR={int(r['year'])}  cat={r['diff_category']:<28}  "
              f"BP: d5={bp_d5} hcd={bp_hcd} delta={r['bp_units_delta']:+.0f}  "
              f"CO: d5={co_d5} hcd={co_hcd} delta={r['co_units_delta']:+.0f}")


B2021-02225 (2650 Telegraph): 2 diff-row(s)
  YEAR=2023  cat=in_both_clean                 BP: d5=45 hcd=45 delta=+0  CO: d5=0 hcd=0 delta=+0
  YEAR=2025  cat=in_both_unit_divergent        BP: d5=0 hcd=0 delta=+0  CO: d5=450 hcd=45 delta=+405

B2022-05117 (2440 Shattuck): 2 diff-row(s)
  YEAR=2023  cat=in_both_clean                 BP: d5=40 hcd=40 delta=+0  CO: d5=0 hcd=0 delta=+0
  YEAR=2025  cat=in_both_unit_divergent        BP: d5=0 hcd=0 delta=+0  CO: d5=200 hcd=40 delta=+160

B2018-05067 (2556 Telegraph): 3 diff-row(s)
  YEAR=2020  cat=hcd_only                      BP: d5=NaN hcd=24 delta=-24  CO: d5=NaN hcd=0 delta=+0
  YEAR=2021  cat=hcd_only                      BP: d5=NaN hcd=24 delta=-24  CO: d5=NaN hcd=0 delta=+0
  YEAR=2022  cat=in_both_tracking_mismatch     BP: d5=0 hcd=NaN delta=+0  CO: d5=0 hcd=NaN delta=+0

B2014-05752 (1698 University): 1 diff-row(s)
  YEAR=2025  cat=in_both_unit_divergent        BP: d5=0 hcd=0 delta=+0  CO: d5=360 hcd=36 delta=+324

B2024-01924 (159

## Cell 10 — Generate methodology_notes.md

Auto-writes the methodology notes to `output/D6/methodology_notes.md`, including the NotebookLM cross-validation that established HCD as a trustworthy oracle.

In [11]:
methodology = """# D6 methodology notes

Auto-generated by 04_reporting/D6_diff_d5_vs_hcd.ipynb. Documents the dedup, join, and unit-counting choices for D6's diff.

## Scope: Table A2 only

D6 compares D5's CPRA-derived BP/CO numbers against HCD's Table A2 (Annual Building Activity). Entitlement-stage data lives in HCD's Table A (Housing Application Activity), which D6 does not currently diff against any D5 equivalent.

Two patterns in Berkeley's CY 2025 Table A submission are documented here as known context for future work:

(1) **California Density Bonus Law base+bonus pairs.** Where a project uses the state density bonus, Berkeley files both the base zoning application and the bonus-enhanced application as separate ZP records with sequential numbering. Reporting both is required by HCD's APR guidance for transparency, but adding both rows would double-count the project (only the bonus version becomes built). Example: 2029 University Ave's ZP2024-0181 (240 units, bonus) and ZP2024-0182 (160 units, base) both appear in CY 2025 Table A. Our HCD CKAN mirror captures the row-level data correctly; v2.projects de-duplicates such pairs to the bonus version.

(2) **CY 2025 doubling extends to Table A.** The same draft+final submission pattern that doubles Table A2's CY 2025 rows also affects Table A: 16 of 32 Berkeley CY 2025 Table A rows are exact duplicates. After dedup using build_hcd_mirror.py's methodology, 16 distinct CY 2025 applications remain with 471 total approved units.

Berkeley's CY 2025 PDF reports 755 approved units as the column total for Table A. This value does not match any straightforward sum of CKAN row-level data (raw sum: 942; full dedup: 471), suggesting the PDF's column total reflects a partial dedup or distinct counting methodology applied between Berkeley's row-level data and its PDF summary. Characterizing the source of this gap requires direct PDF-to-CKAN comparison — out of D6's scope.

A future D7 (Table A diff) would surface both patterns programmatically and reconcile PDF column totals against CKAN row-level data.

## HCD mirror validation against NotebookLM PDF audit

In May 2026, NotebookLM was tasked with reading Berkeley's submitted APR PDFs and producing independent unit counts for CY 2020-2025. Cross-validation against our HCD CKAN mirror showed strong agreement:

| Year | NotebookLM | HCD mirror | Delta |
|------|------------|------------|-------|
| 2020 | 399 | 405 | +6 (+1.5%) |
| 2021 | (PDF unparseable) | 331 | — |
| 2022 | 828 | 828 | 0 (exact) |
| 2023 | 716 | 716 | 0 (exact) |
| 2024 | 708 | 708 | 0 (exact) |
| 2025 | 482 | 481 | -1 (rounding) |

Three of five comparable years match to the unit. CY 2020 is within 1.5%. The CY 2025 dedup methodology (487 → 481) converges independently with NotebookLM's analysis (493 → 482). The CY 2021 gap in NotebookLM's analysis (the PDF was unparseable) is filled by CKAN's structured data.

This cross-source validation establishes the HCD CKAN mirror as a trustworthy oracle for Berkeley's submitted APR totals. The D6 diff therefore compares D5 (CPRA-derived) against a validated reference, not against an unverified source.

## CY 2025 dedup

Berkeley's CY 2025 APR was loaded twice into HCD's CKAN datastore (likely a draft + final submission). 240 of 474 raw Berkeley CY 2025 rows are exact field-copies of another row. Dedup criterion: drop exact duplicates within (APN, STREET_ADDRESS, JURS_TRACKING_ID, BP_ISSUE_DT1, CO_ISSUE_DT1) groups; keep one row per cluster. Audit log at output/D6/dedup_audit_CY2025.csv records every kept/dropped row.

Pre-dedup: 474 rows. Post-dedup: 234 rows. Half of CY 2025 was duplicates.

## Join key normalization

- **APN**: `str.strip().lower()`
- **tracking_id**: `str.strip().upper()` + strip `-REVxx`/`-DEFxx` suffixes (so `B2017-02610-REV01` matches `B2017-02610`)
- **street_address**: `str.strip().lower()` + collapse internal whitespace + strip trailing punctuation

## Date alignment threshold

A `>90 day` BP or CO date difference triggers the `in_both_date_divergent` category. Berkeley's BP/CO dates frequently differ by a few days between D5 (taken from CPRA's Issuance Date / Finaled Date) and HCD (the APR-submitted dates). The 90-day threshold avoids flagging these minor variations while catching cases where Berkeley reports the same permit's BP under different years to the two systems.

## Unit divergence threshold

A `≥1 unit` difference in BP or CO units triggers `in_both_unit_divergent`. Smaller differences (rounding to integer, fractional unit reporting) are absorbed into the `in_both_clean` bucket.

## Tracking mismatch semantics

When D5 and HCD both have rows for the same (APN, year) but with different JURS_TRACKING_ID values, both rows are tagged `in_both_tracking_mismatch`. Most common cause: Berkeley filed two permits at the same APN in the same year. Less common cause: D5's master-permit selection differs from what Berkeley reported to HCD. Most such cases involve HCD rows with NULL tracking_id, which is the dominant noise source.

## in_v2 flag

Each D5 project is tagged `in_v2=True` if its APN matches a row in `databases/berkeley_housing_v2.db.project_parcels`. This identifies the projects the curated v2 list considered worth tracking. The flag does not modify the diff in any way — it's audit metadata for downstream investigation.

## Known unknowns

- **Entitlement data (ENT stage) is not in CPRA.** Planning module is a separate request. D5's entitlement columns are empty; HCD's entitlement data (in Table A, not A2) is not joined.
- **Income-tier breakdown (affordability) is not in CPRA.** D5 places all units in BP_ABOVE_MOD_INCOME by default. HCD has the 11-column tier breakdown. The diff therefore conservatively sums HCD's tier columns for total-units comparison.
- **Tenure is not in CPRA.** D5 defaults to 'Renter'; HCD has actual tenure. The diff doesn't currently compare this.
"""

methodology_path = OUT_DIR / "methodology_notes.md"
methodology_path.write_text(methodology)
print(f"Wrote: {methodology_path.name} ({len(methodology)} chars)")

Wrote: methodology_notes.md (5900 chars)


## Cell 11 — Final report

Print the headline numbers, top 5 unit divergences (post-fix), and 9-address spot-check summary.

In [12]:
total_diff_rows = len(merged)
clean = (merged["diff_category"] == "in_both_clean").sum()
d5_unique = d5["JURS_TRACKING_ID"].nunique()
print(f"Total D5 distinct projects: {d5_unique}")
print(f"Total HCD distinct rows (post-dedup): {len(hcd)}")
print(f"Total diff rows: {total_diff_rows}")
print()
print("Diff category distribution:")
for cat, n in merged["diff_category"].value_counts().items():
    pct = 100 * n / total_diff_rows
    print(f"  {cat:<28}: {n:>5}  ({pct:5.1f}%)")

print()
unit_div = merged[merged["diff_category"] == "in_both_unit_divergent"].copy()
unit_div["max_abs_delta"] = unit_div[["bp_units_delta","co_units_delta"]].abs().max(axis=1)
top5 = unit_div.sort_values("max_abs_delta", ascending=False).head(5)
print("Top 5 largest unit divergences (post-fix):")
for _, r in top5.iterrows():
    print(f"  {r['street_address']} ({int(r['year'])})  tracking={r['d5_tracking']}  "
          f"BP delta={r['bp_units_delta']:+.0f}  CO delta={r['co_units_delta']:+.0f}")

Total D5 distinct projects: 4064
Total HCD distinct rows (post-dedup): 1930
Total diff rows: 6105

Diff category distribution:
  d5_only                     :  3969  ( 65.0%)
  in_both_clean               :   808  ( 13.2%)
  hcd_only                    :   455  (  7.5%)
  in_both_unit_divergent      :   449  (  7.4%)
  in_both_tracking_mismatch   :   423  (  6.9%)
  in_both_date_divergent      :     1  (  0.0%)

Top 5 largest unit divergences (post-fix):
  2150 KITTREDGE St (2024)  tracking=B2021-00008  BP delta=+0  CO delta=+2704
  2000 DWIGHT Way (2025)  tracking=B2021-02404  BP delta=+0  CO delta=+2147
  2001 ASHBY Ave (2025)  tracking=B2021-02905  BP delta=+0  CO delta=+1653
  2099 M L KING JR Way (2024)  tracking=B2021-03950  BP delta=+0  CO delta=+1368
  2527 SAN PABLO Ave (2024)  tracking=B2018-03255  BP delta=+0  CO delta=+441
